# 유사 상권 기반 프랜차이즈 추천

공개용 Notebook입니다. DB credential은 환경변수로 관리합니다.

In [ ]:
import os
from pathlib import Path
import folium
import pandas as pd
import pymysql
import requests
from pyproj import Transformer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = Path(os.getenv('FRANCHISE_INPUT_DATA_DIR', ROOT / 'data'))


In [ ]:
conn = pymysql.connect(
    host=os.environ['FRANCHISE_DB_HOST'],
    port=int(os.getenv('FRANCHISE_DB_PORT', '3306')),
    user=os.environ['FRANCHISE_DB_USER'],
    password=os.environ['FRANCHISE_DB_PASSWORD'],
    db=os.environ['FRANCHISE_DB_NAME'],
)
cursor = conn.cursor()
cursor.execute('SELECT * FROM 상권정보')
rows = cursor.fetchall()
columns = [d[0] for d in cursor.description]
commercial_areas = pd.DataFrame(rows, columns=columns)
target = pd.DataFrame(pd.read_excel(DATA_DIR / '원천동.xlsx'), columns=columns)


## 코사인 유사도 기반 유사 상권 탐색

In [ ]:
IDENTIFIER_COLUMNS = ['상권_구분_코드','상권_코드','상권_코드_명','엑스좌표_값','와이좌표_값','자치구_코드','행정동_코드']
feature_target = target.drop(columns=IDENTIFIER_COLUMNS).fillna(0)
feature_areas = commercial_areas.drop(columns=IDENTIFIER_COLUMNS).fillna(0)
scores = cosine_similarity(feature_target.iloc[[0]], feature_areas)[0]
ranked = commercial_areas[['상권_코드','상권_코드_명']].copy()
ranked['cosine_similarity'] = scores
similar_areas = ranked.sort_values('cosine_similarity', ascending=False).head(10).reset_index(drop=True)
similar_areas


## 유사 상권 좌표 조회 및 지도 시각화

In [ ]:
area_codes = similar_areas['상권_코드'].astype(int).tolist()
placeholders = ','.join(['%s'] * len(area_codes))
cursor.execute(f'SELECT 상권_코드_명, 엑스좌표_값, 와이좌표_값 FROM 상권정보 WHERE 상권_코드 IN ({placeholders})', tuple(area_codes))
coordinates = cursor.fetchall()
m = folium.Map(location=[37.56,126.98], zoom_start=11)
transformer = Transformer.from_crs('EPSG:5181','EPSG:4326',always_xy=True)
for name, x, y in coordinates:
    lon, lat = transformer.transform(x, y)
    folium.Marker([lat, lon], tooltip=name).add_to(m)
m


## 추천 후보 조회
유사 상권의 업종 매출과 프랜차이즈 지표를 결합한 휴리스틱 점수는 sql/sample_queries.sql과 docs/recommendation-design.md를 참고합니다.

In [ ]:
cursor.close()
conn.close()
